In [16]:
from dataclasses import dataclass
import pickle
import gzip
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from time import time

# SIGMOID FUNCTION

In [17]:
def sigmoid(z):
  return 1.0 / (1.0 + np.exp(-z))

In [18]:
def sigmoid_prime(z):
  return sigmoid(z) * (1 - sigmoid(z))

In [19]:
def cost_derivative(output_activations, y):
  return (output_activations - y)

In [20]:
@dataclass
class Network:
  num_layers: int
  biases: list
  weights: list

def init_network(layers):

  return Network(
      len(layers),
      [np.random.randn(y, 1) for y in layers[1:]],
      [np.random.randn(y, x) for x, y in zip(layers[:-1], layers[1:])]
  )

In [21]:
def feedforward(nn,a):
  for b,w in zip(nn.biases, nn.weights):
    a = sigmoid(np.dot(w,a) + b)
  return a

In [22]:
def evaluate(nn, test_data):
  test_results = [(np.argmax(feedforward(nn, x)), y) for (x, y) in test_data]
  return sum(int(x == y) for (x, y) in test_results)

In [23]:
def learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data = None):
    n = len(training_data)

    for j in range(epochs):
        random.shuffle(training_data) # that's where "stochastic" comes from

        mini_batches = [
            training_data[k: k + mini_batch_size] for k in range(0, n, mini_batch_size)
        ]

        for mini_batch in mini_batches:
            stochastic_gradient_descent(nn, mini_batch, learning_rate) # that's where learning really happes

        if test_data:
            print('Epoch {0}: accuracy {1}%'.format(f'{j + 1:2}', 100.0 * evaluate(nn, test_data) / len(test_data)))
        else:
            print('Epoch {0} complete.'.format(f'{j + 1:2}'))

In [24]:
def stochastic_gradient_descent(nn, mini_batch, eta):
    X = np.hstack([x for x, y in mini_batch])  # (784, batch_size)
    Y = np.hstack([y for x, y in mini_batch])  # (10, batch_size)

    nabla_b, nabla_w = backprop_batch(nn, X, Y)

    m = X.shape[1]

    nn.weights = [
        w - (eta / m) * nw
        for w, nw in zip(nn.weights, nabla_w)
    ]

    nn.biases = [
        b - (eta / m) * nb
        for b, nb in zip(nn.biases, nabla_b)
    ]

In [25]:
def backprop_batch(nn, X, Y):

    nabla_b = [np.zeros(b.shape) for b in nn.biases]
    nabla_w = [np.zeros(w.shape) for w in nn.weights]

    activation = X
    activations = [X]
    zs = []

    # Forward pass
    for b, w in zip(nn.biases, nn.weights):
        Z = np.dot(w, activation) + b
        zs.append(Z)

        activation = sigmoid(Z)
        activations.append(activation)

    # Output layer error
    delta = cost_derivative(activations[-1], Y) * sigmoid_prime(zs[-1])

    nabla_b[-1] = np.sum(delta, axis=1, keepdims=True)
    nabla_w[-1] = np.dot(delta, activations[-2].T)

    # Hidden layers
    for l in range(2, nn.num_layers):

        Z = zs[-l]
        sp = sigmoid_prime(Z)

        delta = np.dot(nn.weights[-l + 1].T, delta) * sp

        nabla_b[-l] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-l] = np.dot(delta, activations[-l - 1].T)

    return nabla_b, nabla_w

In [26]:
def load_data():
    f = gzip.open('mnist.pkl.gz', 'rb')
    training_data, validation_data, test_data = pickle.load(f, encoding="bytes")
    f.close()

    return (training_data, validation_data, test_data)

In [27]:
def one_hot_encode(j):
  e = np.zeros((10,1))
  e[j] = 1.0
  return e

In [28]:
def print_shape(name,data):
  print(f'{name} shape: {data.shape}')

In [29]:
def load_data_wrapper():
    tr_d, va_d, te_d = load_data()

    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [one_hot_encode(y) for y in tr_d[1]]
    training_data = zip(training_inputs, training_results)
    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = zip(validation_inputs, va_d[1])
    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = zip(test_inputs, te_d[1])

    return (list(training_data), list(validation_data), list(test_data))

In [30]:
training_data, validation_data, test_data = load_data_wrapper() # load data

nn = init_network([784, 30, 10])

for l in range(0, nn.num_layers - 1):
    print('\nNetwork layer {0}'.format(l + 2)) # disregard the input layer
    print_shape('weights', nn.weights[l])
    print_shape('biases', nn.biases[l])

# hyper parameters
epochs = 15
mini_batch_size = 10
learning_rate = 3.0

print('\nLearning process started...\n')
time_start = time()
learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data)
time_end = time()
time_elapsed = time_end - time_start

print('\nLearning process complete in {0} seconds ({1} seconds per epoch)!\n'.format(f'{time_elapsed:.0f}', f'{time_elapsed / epochs:.1f}'))
print('Validation (with yet unseen data): accuracy {0}%'.format(100.0 * evaluate(nn, validation_data) / len(validation_data)))


Network layer 2
weights shape: (30, 784)
biases shape: (30, 1)

Network layer 3
weights shape: (10, 30)
biases shape: (10, 1)

Learning process started...

Epoch  1: accuracy 90.95%
Epoch  2: accuracy 92.56%
Epoch  3: accuracy 93.25%
Epoch  4: accuracy 93.62%
Epoch  5: accuracy 94.0%
Epoch  6: accuracy 93.8%
Epoch  7: accuracy 94.07%
Epoch  8: accuracy 94.25%
Epoch  9: accuracy 94.35%
Epoch 10: accuracy 94.2%
Epoch 11: accuracy 94.14%
Epoch 12: accuracy 94.57%
Epoch 13: accuracy 94.49%
Epoch 14: accuracy 94.95%
Epoch 15: accuracy 94.79%

Learning process complete in 22 seconds (1.5 seconds per epoch)!

Validation (with yet unseen data): accuracy 95.11%


In [31]:
def load_image(file_name):
    digit = Image.open(file_name)

    # invert, so that background is black (zeros)
    digit = ImageOps.invert(digit)

    pixels = digit.load()

    return np.array(digit).reshape((784, 1)) / 255

In [32]:
def recognize_image(path, file):
    x = load_image(path.format(file))

    y = feedforward(nn, x)

    bitmap = x.reshape((28, 28))

    file_num = int(file)
    result = y.argmax()

    if file_num == result:
        ev = 'correctly'
    else:
        ev = 'incorrectly'

    print(file, 'was', ev, 'recognized as', result)

In [34]:
print('Non-MNIST digits:\n')

for file in range(0,10):
    recognize_image('/content/{0}.png', file)

Non-MNIST digits:

0 was correctly recognized as 0
1 was correctly recognized as 1
2 was correctly recognized as 2
3 was correctly recognized as 3
4 was correctly recognized as 4
5 was incorrectly recognized as 2
6 was correctly recognized as 6
7 was incorrectly recognized as 1
8 was correctly recognized as 8
9 was correctly recognized as 9
